# 02 - Baseline models
Uses the same code path as the production pipeline (`fit_model`), so notebook and pipeline cannot diverge.

In [ ]:
import sys; sys.path.insert(0, "../src")
import pandas as pd, plotly.express as px
from flightops_ml.config import get_paths
from flightops_ml.data.ingestion import load_raw_csv, select_model_frame
from flightops_ml.data.preprocessing import split_dataset
from flightops_ml.data.validation import build_quality_report
from flightops_ml.config import load_training_config
from flightops_ml.training.train import fit_model
from flightops_ml.evaluation.metrics import classification_metrics
cfg = load_training_config()
X, y = select_model_frame(load_raw_csv(get_paths().dataset_path()))
sp = split_dataset(X, y, cfg['seed'])

In [ ]:
rows = {}
for name in ['logistic_regression','random_forest','xgboost']:
    pipe, hist, _ = fit_model(name, cfg, sp, cfg['seed'])
    rows[name] = classification_metrics(sp.y_val, pipe.predict_proba(sp.X_val)[:,1], 0.5)  # validation only
pd.DataFrame(rows).T[['precision','recall','f1','pr_auc','roc_auc']]

Note: this notebook scores on the **validation** split only. The test split is reserved for the pipeline's final evaluation.